# [9.4] White-box Evals and Monitors

> **Notebooks: [exercises](../../exercises/part4_white_box_evals_monitors/9.4_White-box_Evals_and_Monitors_exercises.ipynb) | [solutions](../../exercises/part4_white_box_evals_monitors/9.4_White-box_Evals_and_Monitors_solutions.ipynb)**


In [ ]:
GT_TIER = "GT-3"
EXERCISE_ID = "9_4_white_box_evals_and_monitors"
EXPECTED_RUNTIME = "45-60 minutes for exercises; a few minutes for the pinned Pythia CUDA preflight"
REQUIRES_GPU = True



> **Local-first extension.** This section builds white-box monitor reports from
> safe records, then verifies the full path with a pinned Pythia-70M hidden-state
> preflight. The real-model path uses hidden states and next-token logits only;
> it does not generate completions and does not claim a broad deployment monitor.

Please send any problems / bugs on the `#errata` channel in the [Slack group](https://info-arena.github.io/ARENA_img/slack.html), and ask any questions on the dedicated channels for this chapter of material.

If you want to change to dark mode, you can do this by clicking the three horizontal lines in the top-right, then navigating to Settings -> Theme.

Links to earlier chapters: [(0) Fundamentals](https://arena-chapter0-fundamentals.streamlit.app/), [(1) Transformer Interpretability](https://arena-chapter1-transformer-interp.streamlit.app/), [(2) RL](https://arena-chapter2-rl.streamlit.app/), [(3) LLM Evaluations](https://arena-chapter3-llm-evals.streamlit.app/), [(4) Alignment Science](https://arena-chapter4-alignment-science.streamlit.app/).

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/headers/header-14-1.png" width="350">


# Introduction

Black-box evals are useful, but they can miss failures that are visible in the
model's internals before they show up in a coarse behavior proxy. The hard part
is not just finding a high-scoring feature. The hard part is making the monitor
auditable:

```text
what did the reviewer see?
which internal evidence fired?
does the score calibrate on held-out records?
which black-box misses were caught?
were false positives documented?
do feature explanations predict held-out labels?
```

In this section, you will build a small monitor-reporting stack and then inspect
a pinned CUDA result. The student-facing exercises use deterministic tensors so
you can reason about every field. The final report uses Pythia-70M hidden states
on safe generated monitor records.

<img src="../assets/white_box_evals_monitors_validation_loop.svg" width="860">

The core question is:

```text
Can a simple white-box hidden-state direction catch held-out safe failure-policy
records that a next-token pass/fail proxy misses, while failing label-shuffled
and random-direction controls?
```

This is deliberately narrower than a deployment monitor. It is a course preflight
for the evidence pattern: calibrated internal scores, explicit black-box
comparison, false-positive notes, held-out explanation validation, and negative
controls.


## Reading Material

The useful mental model is the one you have built throughout Chapter 9:

```text
behavior-only evidence can tell you that something happened;
white-box evidence can sometimes tell you where to look;
controls decide whether the white-box story is trustworthy.
```

The report-backed result is best described as:

```text
Pythia-70M safe white-box monitor hidden-state preflight.
```

Do not describe it as:

```text
a deployment-ready safety monitor;
a broad harmful-content detector;
a generated-completion benchmark;
proof that white-box monitors always beat black-box evals.
```


## Content & Learning Objectives

### 1. Monitor dashboard rows

You will keep the visible behavior next to the internal features and scores that
made the monitor fire.

> ##### Learning Objectives
>
> * Preserve reviewer-visible fields.
> * Store active feature names immutably.
> * Reject non-finite monitor scores before they enter a report.

### 2. Calibration by AUROC

You will implement binary AUROC directly, including ties.

> ##### Learning Objectives
>
> * Interpret AUROC as a positive-negative pairwise ranking probability.
> * Reject empty, non-finite, one-class, or non-binary evidence.
> * Treat calibration thresholds as explicit unit-interval gates.

### 3. White-box versus black-box catches

You will count only the failures that the white-box monitor catches and the
black-box proxy misses.

> ##### Learning Objectives
>
> * Distinguish true positives from white-box-only true positives.
> * Keep caught-example indices for review.
> * Avoid silently coercing non-binary predictions.

### 4. False positives and explanations

You will require reviewer notes for false positives and validate explanations on
held-out labels.

> ##### Learning Objectives
>
> * Treat false positives as audit objects, not harmless noise.
> * Use held-out labels for explanation validation.
> * Keep explanation validation separate from the monitor decision itself.


## Setup code


In [ ]:
import json
import sys
from dataclasses import dataclass
from pathlib import Path

import torch as t

chapter = "chapter9_alignment_interpretability"
section = "part4_white_box_evals_monitors"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part4_white_box_evals_monitors.tests as tests
import part4_white_box_evals_monitors.utils as utils

MAIN = __name__ == "__main__"


In [ ]:
@dataclass(frozen=True)
class MonitorDashboardRow:
    prompt: str
    model_output: str
    active_features: tuple[str, ...]
    refusal_score: float
    hallucination_score: float
    cot_faithfulness_score: float


@dataclass(frozen=True)
class MonitorCalibrationReport:
    auroc: float
    calibrated: bool


@dataclass(frozen=True)
class MissedFailureReport:
    caught_failure_indices: tuple[int, ...]
    num_caught_failures: int
    catches_black_box_miss: bool


@dataclass(frozen=True)
class FalsePositiveDocumentationReport:
    false_positive_indices: tuple[int, ...]
    num_false_positives: int
    documented: bool


@dataclass(frozen=True)
class FeatureExplanationValidationReport:
    heldout_accuracy: float
    explanations_validated: bool


In [ ]:
def _require_finite_tensor(name: str, tensor: t.Tensor) -> None:
    if tensor.numel() == 0:
        raise ValueError(f"{name} must be non-empty.")
    if not t.isfinite(tensor.float()).all():
        raise ValueError(f"{name} must contain only finite values.")


def _require_finite_scalar(name: str, value: float) -> None:
    value_tensor = t.tensor(value, dtype=t.float32)
    if not t.isfinite(value_tensor):
        raise ValueError(f"{name} must be finite.")


def _require_unit_interval(name: str, value: float) -> None:
    _require_finite_scalar(name, value)
    if not 0.0 <= value <= 1.0:
        raise ValueError(f"{name} must be between 0 and 1.")


def _require_binary_tensor(name: str, tensor: t.Tensor) -> None:
    _require_finite_tensor(name, tensor)
    if tensor.dtype == t.bool:
        return
    values_are_binary = tensor.eq(0) | tensor.eq(1)
    if not values_are_binary.all():
        raise ValueError(f"{name} must contain only binary 0/1 values.")




# Dashboard Rows

### Exercise 1 - build one monitor dashboard row

> Difficulty: easy
> Importance: high
>
> You should spend 5 minutes on this exercise.

A monitor dashboard row is not just a score. It is the unit a reviewer can
audit later: the prompt, the visible output, the active feature labels, and the
scores that made the row suspicious.


In [ ]:
def monitor_dashboard_row(
    *,
    prompt: str,
    model_output: str,
    active_features: list[str],
    refusal_score: float,
    hallucination_score: float,
    cot_faithfulness_score: float,
) -> MonitorDashboardRow:
    raise NotImplementedError()


tests.test_monitor_dashboard_row_preserves_review_fields(monitor_dashboard_row)



<details>
<summary>Expected output</summary>

```text
All tests in `test_monitor_dashboard_row_preserves_review_fields` passed!
```

</details>

<details>
<summary>Help - what belongs in a dashboard row?</summary>

The row should be enough to reconstruct why the monitor fired. If you return
only scores, a later reviewer cannot tell whether the monitor flagged a real
problem, a benign style feature, or a missing-context artifact.

</details>

<details>
<summary>Common bugs</summary>

- Dropping the prompt or output and returning only internal scores.
- Leaving `active_features` as a mutable list.
- Accepting `nan` scores, which can poison later calibration tables.

</details>

<details>
<summary>Solution</summary>

```python
def monitor_dashboard_row(
    *,
    prompt: str,
    model_output: str,
    active_features: list[str],
    refusal_score: float,
    hallucination_score: float,
    cot_faithfulness_score: float,
) -> MonitorDashboardRow:
    if not prompt.strip():
        raise ValueError("prompt must be non-empty.")
    if not model_output.strip():
        raise ValueError("model_output must be non-empty.")
    active_features_tuple = tuple(active_features)
    if not all(isinstance(feature, str) and feature.strip() for feature in active_features_tuple):
        raise ValueError("active_features must contain non-empty feature names.")
    for name, value in {
        "refusal_score": refusal_score,
        "hallucination_score": hallucination_score,
        "cot_faithfulness_score": cot_faithfulness_score,
    }.items():
        _require_finite_scalar(name, value)

    return MonitorDashboardRow(
        prompt=prompt,
        model_output=model_output,
        active_features=active_features_tuple,
        refusal_score=refusal_score,
        hallucination_score=hallucination_score,
        cot_faithfulness_score=cot_faithfulness_score,
    )
```

</details>


# Calibration

### Exercise 2 - implement binary AUROC

> Difficulty: medium
> Importance: high
>
> You should spend 10 minutes on this exercise.

For binary labels, AUROC is the probability that a randomly chosen positive
example has a higher score than a randomly chosen negative example. Ties count
as half a win.


In [ ]:
def binary_auroc(scores: t.Tensor, labels: t.Tensor) -> float:
    raise NotImplementedError()


tests.test_binary_auroc_counts_ties_and_validates_inputs(binary_auroc)



<details>
<summary>Expected output</summary>

```text
All tests in `test_binary_auroc_counts_ties_and_validates_inputs` passed!
```

</details>

<details>
<summary>Help - why pairwise AUROC?</summary>

Accuracy depends on a threshold. AUROC asks whether the score ranks failures
above clean examples before you choose a threshold. That is a better first
calibration check for a monitor score.

</details>

<details>
<summary>Common bugs</summary>

- Computing threshold accuracy instead of pairwise ranking.
- Ignoring tied positive-negative pairs.
- Calling `.bool()` on labels before checking that they are actually binary.

</details>

<details>
<summary>Solution</summary>

```python
def binary_auroc(scores: t.Tensor, labels: t.Tensor) -> float:
    scores = scores.flatten().float()
    labels = labels.flatten()
    _require_finite_tensor("scores", scores)
    _require_binary_tensor("labels", labels)
    if scores.shape != labels.shape:
        raise ValueError("scores and labels must have matching shape.")
    labels = labels.bool()

    positive_scores = scores[labels]
    negative_scores = scores[~labels]
    if positive_scores.numel() == 0 or negative_scores.numel() == 0:
        raise ValueError("both positive and negative labels are required.")

    comparisons = positive_scores[:, None] - negative_scores[None, :]
    wins = comparisons.gt(0).float().sum().item()
    ties = comparisons.eq(0).float().sum().item()
    total_pairs = positive_scores.numel() * negative_scores.numel()
    return (wins + 0.5 * ties) / total_pairs
```

</details>


### Exercise 3 - calibrate monitor scores

> Difficulty: medium
> Importance: high
>
> You should spend 10 minutes on this exercise.

Calibration is a gate, not a decoration. A monitor that cannot rank held-out
failure records above clean records should not be accepted.


In [ ]:
def monitor_calibration_report(
    monitor_scores: t.Tensor,
    failure_labels: t.Tensor,
    *,
    min_auroc: float = 0.8,
) -> MonitorCalibrationReport:
    raise NotImplementedError()


tests.test_monitor_calibration_report_matches_reference(monitor_calibration_report)



<details>
<summary>Expected output</summary>

```text
All tests in `test_monitor_calibration_report_matches_reference` passed!
```

</details>

<details>
<summary>Help - what does calibrated mean here?</summary>

It means only that the score ranks this held-out set well enough under the
declared threshold. It does not mean probabilities are calibrated, and it does
not mean the monitor will generalize to new tasks.

</details>

<details>
<summary>Common bugs</summary>

- Returning the AUROC float instead of a report dataclass.
- Treating lower scores as better without documenting the sign convention.
- Allowing `min_auroc` values outside `[0, 1]`.

</details>

<details>
<summary>Solution</summary>

```python
def monitor_calibration_report(
    monitor_scores: t.Tensor,
    failure_labels: t.Tensor,
    *,
    min_auroc: float = 0.8,
) -> MonitorCalibrationReport:
    _require_unit_interval("min_auroc", min_auroc)
    auroc = binary_auroc(monitor_scores, failure_labels)
    return MonitorCalibrationReport(
        auroc=auroc,
        calibrated=auroc >= min_auroc,
    )
```

</details>


# White-box Versus Black-box

### Exercise 4 - find failures missed by a black-box proxy

> Difficulty: medium
> Importance: high
>
> You should spend 10 minutes on this exercise.

The useful count is not "how many failures did the white-box monitor catch?"
It is "how many failures did the white-box monitor catch that this black-box
proxy missed?"


In [ ]:
def missed_failure_report(
    white_box_predictions: t.Tensor,
    black_box_predictions: t.Tensor,
    failure_labels: t.Tensor,
) -> MissedFailureReport:
    raise NotImplementedError()


tests.test_missed_failure_report_identifies_white_box_only_catches(missed_failure_report)



<details>
<summary>Expected output</summary>

```text
All tests in `test_missed_failure_report_identifies_white_box_only_catches` passed!
```

</details>

<details>
<summary>Help - why keep caught indices?</summary>

Aggregate counts are not enough for monitor debugging. The examples caught by
white-box and missed by black-box are exactly the rows a reviewer should inspect
first.

</details>

<details>
<summary>Common bugs</summary>

- Counting every white-box true positive.
- Comparing predictions without checking shapes.
- Silently converting `2` or `-1` to `True` with `.bool()`.

</details>

<details>
<summary>Solution</summary>

```python
def missed_failure_report(
    white_box_predictions: t.Tensor,
    black_box_predictions: t.Tensor,
    failure_labels: t.Tensor,
) -> MissedFailureReport:
    white = white_box_predictions.flatten()
    black = black_box_predictions.flatten()
    labels = failure_labels.flatten()
    _require_binary_tensor("white_box_predictions", white)
    _require_binary_tensor("black_box_predictions", black)
    _require_binary_tensor("failure_labels", labels)
    if white.shape != black.shape or white.shape != labels.shape:
        raise ValueError("prediction and label tensors must have matching shape.")

    white = white.bool()
    black = black.bool()
    labels = labels.bool()
    caught_mask = labels & white & ~black
    indices = tuple(int(index.item()) for index in caught_mask.nonzero().flatten())
    return MissedFailureReport(
        caught_failure_indices=indices,
        num_caught_failures=len(indices),
        catches_black_box_miss=len(indices) > 0,
    )
```

</details>


# False Positives And Explanations

### Exercise 5 - document every false positive

> Difficulty: easy
> Importance: high
>
> You should spend 5 minutes on this exercise.

False positives are not harmless bookkeeping. If a monitor fires on safe
behavior, the report should keep a written note for each case.


In [ ]:
def false_positive_documentation_report(
    monitor_predictions: t.Tensor,
    failure_labels: t.Tensor,
    documentation: dict[int, str] | None = None,
) -> FalsePositiveDocumentationReport:
    raise NotImplementedError()


tests.test_false_positive_documentation_requires_notes(
    false_positive_documentation_report,
)



<details>
<summary>Expected output</summary>

```text
All tests in `test_false_positive_documentation_requires_notes` passed!
```

</details>

<details>
<summary>Help - why documentation is part of the metric</summary>

A monitor with zero false positives and a monitor with undocumented false
positives are very different. The first may be clean on this set; the second
has reviewer work outstanding.

</details>

<details>
<summary>Common bugs</summary>

- Accepting blank strings as documentation.
- Indexing documentation before flattening the prediction tensor.
- Forgetting that zero false positives should pass vacuously.

</details>

<details>
<summary>Solution</summary>

```python
def false_positive_documentation_report(
    monitor_predictions: t.Tensor,
    failure_labels: t.Tensor,
    documentation: dict[int, str] | None = None,
) -> FalsePositiveDocumentationReport:
    predictions = monitor_predictions.flatten()
    labels = failure_labels.flatten()
    _require_binary_tensor("monitor_predictions", predictions)
    _require_binary_tensor("failure_labels", labels)
    if predictions.shape != labels.shape:
        raise ValueError("predictions and labels must have matching shape.")

    predictions = predictions.bool()
    labels = labels.bool()
    false_positive_mask = predictions & ~labels
    indices = tuple(int(index.item()) for index in false_positive_mask.nonzero().flatten())
    documentation = documentation or {}
    documented = all(bool(documentation.get(index, "").strip()) for index in indices)
    return FalsePositiveDocumentationReport(
        false_positive_indices=indices,
        num_false_positives=len(indices),
        documented=documented,
    )
```

</details>


### Exercise 6 - validate feature explanations on held-out labels

> Difficulty: medium
> Importance: high
>
> You should spend 10 minutes on this exercise.

Feature explanations are only useful if they predict something out of sample.
Here the explanation-derived predictions must match held-out failure labels.


In [ ]:
def feature_explanation_validation_report(
    explanation_predictions: t.Tensor,
    heldout_labels: t.Tensor,
    *,
    min_accuracy: float = 0.8,
) -> FeatureExplanationValidationReport:
    raise NotImplementedError()


tests.test_feature_explanation_validation_uses_heldout_accuracy(
    feature_explanation_validation_report,
)



<details>
<summary>Expected output</summary>

```text
All tests in `test_feature_explanation_validation_uses_heldout_accuracy` passed!
```

</details>

<details>
<summary>Help - do not validate explanations on themselves</summary>

The explanation should make a prediction about held-out examples. If you reuse
the monitor predictions as the explanation target, the check becomes circular.

</details>

<details>
<summary>Common bugs</summary>

- Validating explanations on the examples used to choose the feature.
- Returning raw equality tensors instead of a scalar accuracy.
- Accepting invalid `min_accuracy` thresholds.

</details>

<details>
<summary>Solution</summary>

```python
def feature_explanation_validation_report(
    explanation_predictions: t.Tensor,
    heldout_labels: t.Tensor,
    *,
    min_accuracy: float = 0.8,
) -> FeatureExplanationValidationReport:
    _require_unit_interval("min_accuracy", min_accuracy)
    predictions = explanation_predictions.flatten()
    labels = heldout_labels.flatten()
    _require_binary_tensor("explanation_predictions", predictions)
    _require_binary_tensor("heldout_labels", labels)
    if predictions.shape != labels.shape:
        raise ValueError("predictions and labels must have matching shape.")

    predictions = predictions.bool()
    labels = labels.bool()
    accuracy = predictions.eq(labels).float().mean().item()
    return FeatureExplanationValidationReport(
        heldout_accuracy=accuracy,
        explanations_validated=accuracy >= min_accuracy,
    )
```

</details>


# Notebook Contract

After implementing the reports, assemble the smoke contract used by the visible
notebook tests.


In [ ]:
def run_smoke_test(cpu: bool = True) -> dict:
    _ = cpu
    return {
        "dashboard": monitor_dashboard_row(
            prompt="Summarize this harmless note.",
            model_output="A concise summary.",
            active_features=["summary", "benign"],
            refusal_score=0.1,
            hallucination_score=0.2,
            cot_faithfulness_score=0.9,
        ).__dict__,
        "calibration": monitor_calibration_report(
            t.tensor([0.1, 0.4, 0.8, 0.9]),
            t.tensor([0, 0, 1, 1], dtype=t.bool),
            min_auroc=0.9,
        ).__dict__,
        "missed_failure": missed_failure_report(
            t.tensor([1, 0, 0], dtype=t.bool),
            t.tensor([0, 0, 0], dtype=t.bool),
            t.tensor([1, 1, 0], dtype=t.bool),
        ).__dict__,
        "false_positive": false_positive_documentation_report(
            t.tensor([1, 0, 1], dtype=t.bool),
            t.tensor([1, 0, 0], dtype=t.bool),
            documentation={2: "Benign style feature caused a high monitor score."},
        ).__dict__,
        "explanation_validation": feature_explanation_validation_report(
            t.tensor([1, 0, 1, 0], dtype=t.bool),
            t.tensor([1, 0, 1, 0], dtype=t.bool),
            min_accuracy=1.0,
        ).__dict__,
    }


tests.test_notebook_contract(run_smoke_test)



<details>
<summary>Expected output</summary>

```text
All tests in `test_notebook_contract` passed!
```

</details>

<details>
<summary>Common bugs</summary>

- Returning dataclass objects rather than JSON-serializable dictionaries.
- Forgetting one monitor gate in the smoke contract.
- Using a false-positive fixture without documentation.

</details>


# Signature Result

The real-model result is intentionally narrow but useful: a hidden-state monitor
direction separates the held-out safe failure-policy records, catches failures
missed by a next-token pass/fail proxy, and fails the label-shuffle and
random-direction controls.

<img src="../assets/white_box_evals_monitors_signature_result.svg" width="860">

| Field | Value |
|---|---:|
| Model | `EleutherAI/pythia-70m-deduped` |
| Revision | `e93a9faa9c77e5d09219f6c868bfc7a1bd65593c` |
| Train / held-out prompts | `36 / 24` |
| Hidden state shape | `[24, 512]` |
| Monitor AUROC | `1.000` |
| White-box accuracy | `1.000` |
| Black-box proxy accuracy | `0.875` |
| Black-box missed failures caught | `3` |
| Label-shuffled monitor AUROC | `0.388` |
| Random-direction monitor AUROC | `0.088` |
| Held-out explanation accuracy | `1.000` |
| Forced false-positive notes | `4 / 4 documented` |
| Peak VRAM | `0.309 GB` |
| Generated completions | `false` |

<details>
<summary>Interpreting the result</summary>

The white-box monitor is not just high accuracy on the same examples used to
fit it. The report uses held-out contexts, compares against a black-box proxy,
and includes controls that should fail. The important educational result is the
shape of the audit, not the claim that this tiny monitor is deployment-ready.

</details>

<details>
<summary>What would falsify this claim?</summary>

The claim should fail if the hidden-state direction does not calibrate on
held-out records, if label-shuffled or random directions also calibrate, if the
black-box proxy catches every failure the white-box monitor catches, if false
positives lack notes, or if explanation validation reuses the monitor decision
instead of held-out labels.

</details>


# Full CUDA Verification Path

The committed report for this section checks the full local path:

1. Load pinned `EleutherAI/pythia-70m-deduped` at revision
   `e93a9faa9c77e5d09219f6c868bfc7a1bd65593c`.
2. Build 36 train and 24 held-out safe monitor records across five benign
   failure-policy kinds.
3. Extract final-token hidden states and next-token logits on CUDA.
4. Train a thresholded hidden-state monitor direction.
5. Compare with a real next-token `pass`/`fail` proxy.
6. Require calibration, a missed black-box failure, false-positive
   documentation, held-out explanation accuracy, and two negative controls.


In [ ]:
def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    from part4_white_box_evals_monitors.solutions import (
        run_pythia_white_box_monitor_preflight,
    )

    return run_pythia_white_box_monitor_preflight(max_vram_gb=max_vram_gb)


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    return run_gpu_test(max_vram_gb=max_vram_gb)


In [ ]:
report = json.loads((section_dir / "verification_report.json").read_text())
gpu_result = report["metrics"]["gpu_test"]
tests.test_committed_gpu_report_matches_white_box_monitor_contract(gpu_result)



<details>
<summary>Expected output</summary>

```text
All tests in `test_committed_gpu_report_matches_white_box_monitor_contract` passed!
```

</details>

You can regenerate the report locally with:

```bash
BNB_CUDA_VERSION=130 uv run python scripts/run_extension_verification_reports.py --section 9.4 --max-vram-gb 24.0
```


# Limitations

This section does not prove that a real deployment monitor is solved. It does
not use harmful prompts, generated completions, live traffic, or a broad
red-team benchmark. The black-box baseline is a deliberately small next-token
`pass`/`fail` proxy, so beating it is evidence for this preflight only.

The explanation-validation check uses the held-out eval-record taxonomy. That
is a useful anti-circularity check, but it is still much easier than explaining
natural deployment failures. Treat it as a training-wheel version of the audit
you would want for real monitors.


# Further Research

- Replace the next-token proxy with a stronger black-box judge and ask whether
  the white-box monitor still catches anything unique.
- Add a real feature dictionary or crosscoder feature source, then rerun the
  false-positive and explanation-validation gates.
- Stress-test calibration under context families that were not used in the
  train or held-out split.
- Build a dashboard that renders caught misses and false positives as reviewer
  rows, not just aggregate metrics.
